# Evaluating deep neural network 

 In the first exercise we built, trained and regularised a DNN on `fashion-mnist` and judged it mainly by accuracy. Accuracy is a good starting point, however on its own it can hide serious problems. Here we learn how to measure how and where a model is right or wrong.

We will keep using Keras and `fashion-mnist`, and add the [scikit-learn metrics API](https://scikit-learn.org/stable/modules/model_evaluation.html), which makes it easy to compute almost any metric from a model's predictions.

By the end of this notebook, users will practice:

- Defining true/false positives and negatives.
- Reading a confusion matrix and a per-class classification report.
- Using probabilities for ROC curves and decision thresholds.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             top_k_accuracy_score)


import numpy as np
import matplotlib.pyplot as plt
plt.style.use('ggplot')

print('Using TensorFlow v%s' % tf.__version__)

---

# Train a deep neural model to evaluate

We now return to `fashion-mnist` with 10 classes. We reuse the network from introdcution to deep neural network, with two changes:

1. The output layer uses `softmax` instead of `sigmoid`, so the 10 outputs are probabilities that sum to 1. This is the standard choice for multi-class classification and is what the metrics below expect.
2. We hold out 10% of the training data for validation (`validation_split=0.1`), so the test set stays completely unseen until the final evaluation. 

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

train_images = train_images / 255.0
test_images = test_images / 255.0


string_labels = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


print("Test images per class:", np.bincount(test_labels))

The test set has exactly 1000 images per class, so this dataset is **balanced**. Keep this in mind: it makes accuracy more trustworthy here than in the disease example, and it will matter when we compare averages later.

In [ ]:
model = Sequential()
model.add(Flatten(input_shape=(28, 28)))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(10, activation='softmax'))


model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

training_history = model.fit(train_images, train_labels, epochs=10, batch_size=32,
                             validation_split=0.1)

---

Scikit-learn metrics compare the true labels with the predicted labels. A Keras model, however, outputs probabilities, so we need one extra step.

`model.predict()` returns an array with one row per image and one column per class. Each row holds the probability of each class, and the predicted class is the column with the highest probability (`argmax`).

In [ ]:
probs = model.predict(test_images)

pred_labels = probs.argmax(axis=1)

print("Probabilities:", np.round(probs[0], 3))
print("Sum of probabilities: %.3f" % probs[0].sum())
print("Predicted class: %d (%s)" % (pred_labels[0], string_labels[pred_labels[0]]))
print("True class:      %d (%s)" % (test_labels[0], string_labels[test_labels[0]]))

`model.evaluate()` and `accuracy_score()` should give the same answer, since both compare predictions with the true labels:

In [ ]:
_, test_acc = model.evaluate(test_images, test_labels, verbose=0)
print("Accuracy from model.evaluate(): %.4f" % test_acc)

print("Accuracy from accuracy_score(): %.4f" % accuracy_score(test_labels, pred_labels))

---

Accuracy tells us how often*the model is wrong, but not where. The confusion matrix counts, for every pair of classes, how many images of the true class were predicted as each class:

- each row is the true class;
- each column is the predicted class;
- the diagonal holds the correct predictions, and everything off the diagonal is a mistake.

In [ ]:
cm = confusion_matrix(test_labels, pred_labels)


def plot_confusion_matrix(cm, class_names, title, fmt='d', ax=None):
    ax = ax if ax is not None else plt.gca()
    im = ax.imshow(cm, cmap=plt.cm.Blues)
    ax.set_title(title)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.grid(False)
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha='center', va='center', fontsize=8,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    return im

cm_norm = cm / cm.sum(axis=1, keepdims=True)
plt.figure(dpi=100, figsize=(16, 7))
plt.subplot(1, 2, 1)
plot_confusion_matrix(cm, string_labels, 'Confusion matrix (counts)')
plt.subplot(1, 2, 2)
plot_confusion_matrix(cm_norm, string_labels, 'Confusion matrix (normalised by row)', fmt='.2f')
plt.tight_layout()
plt.show()

The right hand plot is normalised by row, each row sums to 1, so the diagonal entry is the fraction of that class that was found correctly.

In [ ]:
cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)

print("Most frequent mistakes:")
for flat_index in np.argsort(cm_errors.ravel())[::-1][:5]:
    true_class, pred_class = np.unravel_index(flat_index, cm_errors.shape)
    print("  %-10s predicted as %-10s: %d times" % (string_labels[true_class],
                                                    string_labels[pred_class],
                                                    cm_errors[true_class, pred_class]))

---

The function `classification_report()` computes all of these at once. The column `support` is the number of true images of each class.

In [ ]:
print(classification_report(test_labels, pred_labels, target_names=string_labels, digits=3))

In [ ]:
precision_per_class = precision_score(test_labels, pred_labels, average=None)
recall_per_class = recall_score(test_labels, pred_labels, average=None)
f1_per_class = f1_score(test_labels, pred_labels, average=None)


x = np.arange(len(string_labels))
width = 0.27
plt.figure(dpi=100, figsize=(12, 4))
plt.bar(x - width, precision_per_class, width, label='Precision')
plt.bar(x, recall_per_class, width, label='Recall')
plt.bar(x + width, f1_per_class, width, label='F1')
plt.xticks(x, string_labels, rotation=45, ha='right')
plt.ylim(0.4, 1.0)
plt.legend(loc='lower left')
plt.title("Per-class metrics")
plt.show()

# the weakest class according to F1
worst = f1_per_class.argmin()
print("Lowest F1: %s (precision %.3f, recall %.3f, F1 %.3f)" % (
    string_labels[worst], precision_per_class[worst], recall_per_class[worst], f1_per_class[worst]))

---

The ROC curve and its area  summarise how well the probabilities rank the true class above the others, independently of any particular decision threshold. An AUC of 1.0 is perfect ranking and 0.5 is no better than guessing. For multiple classes, scikit-learn can compute it one class against all the rest (`multi_class='ovr'`):

In [ ]:
auc = roc_auc_score(test_labels, probs, multi_class='ovr')
print("ROC AUC (one-vs-rest, macro average): %.4f" % auc)

---

In [ ]:
shirt = string_labels.index('Shirt')
y_shirt = (test_labels == shirt).astype(int)
score_shirt = probs[:, shirt]


precisions, recalls, thresholds = precision_recall_curve(y_shirt, score_shirt)

# ROC curve
fpr, tpr, _ = roc_curve(y_shirt, score_shirt)

plt.figure(dpi=100, figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.xlabel('Threshold on P(Shirt)')
plt.legend()
plt.title("Precision and recall vs threshold")

plt.subplot(1, 2, 2)
plt.plot(fpr, tpr, label='AUC = %.3f' % roc_auc_score(y_shirt, score_shirt))
plt.plot([0, 1], [0, 1], 'k--', label='Guessing')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate (recall)')
plt.legend()
plt.title("ROC curve for Shirt")
plt.show()

In [ ]:
print("Threshold  Precision  Recall")
for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    y_hat = (score_shirt >= threshold).astype(int)
    print("   %.1f       %.3f     %.3f" % (threshold,
                                           precision_score(y_shirt, y_hat),
                                           recall_score(y_shirt, y_hat)))